In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
df1 = pd.read_csv("yere1.csv")
df2 = pd.read_csv("yere2.csv")

df = pd.concat([df1, df2], ignore_index=True)

In [ ]:
df.shape

In [ ]:
m = df.isnull().sum()
p = m/len(df)*100
p

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'])

In [ ]:
df.severerisk.isna().sum()

In [ ]:
df.set_index('datetime', inplace=True)

In [ ]:
df.columns

In [ ]:
# Resample the data to weekly means
weekly_means = df.resample('W').mean(numeric_only=True)
df = weekly_means

In [ ]:
df

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['temp'], label='Average Temperature', color='green')
plt.title('Temperature Over Time')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid()
plt.show()

In [ ]:
model_add = seasonal_decompose(df['temp'], model='additive',period = 52)
fig = model_add.plot()
fig.set_size_inches((12, 7))
fig.tight_layout()
plt.show()

In [ ]:
df.drop(["tempmax", "tempmin", "feelslikemax", "snow", "feelslikemin", "severerisk", "feelslike", "solarenergy", "snowdepth", "windgust", "moonphase"], axis=1, inplace=True)

numeric_cols = df.select_dtypes(include=np.number).columns

The following columns were mainly dropped for different  reasons; some were directly related to the mean temp, others were irrelevant to the target (both from general and knwoledge specific way)

In [ ]:
n = df.select_dtypes(include=np.number)
plt.figure(figsize=(15, 12))
correlation_matrix = n.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")

In [ ]:
df_monthly = df['temp'].resample('M').mean()
df_weekly = df['temp'].resample('W').mean()

plt.figure(figsize=(12,5))
plt.plot(df_monthly, label='Monthly Avg Temp')
plt.plot(df_weekly, label='Weekly Avg Temp', alpha=0.7)
plt.title('Aggregated Temperature Trends')
plt.legend()
plt.show()

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows*4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.scatterplot(x=df[col], y=df['temp'], ax=axes[i])
    axes[i].set_title(f'{col} vs Temp')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ADF Test

ts = df.sort_values("datetime")["temp"].dropna()

adf_result = adfuller(df['temp'])
print("ADF Test Results:")
print(f'ADF Statistic: {adf_result[0]}')
print(f'p-value: {adf_result[1]}')
print(f'Critical Values: {adf_result[4]}')

# KPSS Test
kpss_result = kpss(df['temp'], regression='c')
print("\nKPSS Test Results:")
print(f'KPSS Statistic: {kpss_result[0]}')
print(f'p-value: {kpss_result[1]}')
print(f'Critical Values: {kpss_result[3]}')

if adf_result[1] < 0.05 and kpss_result[1] > 0.05:
    print('The time series data is stationary based on ADF and KPSS tests.')
elif adf_result[1] >= 0.05 and kpss_result[1] < 0.05:
    print('The time series data is non-stationary based on both ADF and KPSS tests.')
elif adf_result[1] >= 0.05 and kpss_result[1] >= 0.05:
    print('The time series data is non-stationary based on ADF test, but stationary based on KPSS test.')
else:
    print('The time series data is inconclusive for stationarity based on ADF and KPSS tests.')

In [ ]:
df.reset_index(inplace = True)

In [ ]:
# Plot ACF
plt.figure(figsize=(12, 6))
plot_acf(df["temp"], lags=52, alpha=0.05)
plt.title('Autocorrelation Function (ACF) for Temperature')
plt.xlabel('Lags')
plt.ylabel('Autocorrelation')
plt.show()

# Plot PACF
plt.figure(figsize=(12, 6))
plot_pacf(df["temp"], lags=52, alpha=0.05)
plt.title('Partial Autocorrelation Function (PACF) for Temperature')
plt.xlabel('Lags')
plt.ylabel('Partial Autocorrelation')
plt.show()

From PACF we can see that the first two lags are significant, while the rest is closer to 0, so it is insignificant.
 But note that ACF can be influenced by trends and seasonality, and we should pay more attention to the results obtained statistical tests like ADF and KPSS

In [ ]:
df.to_csv("yerevan_weather.csv", index=False)    